<p align="center">
<a href="https://duckietown.com"><img src="../assets/images/dtlogo.png" alt="Duckietown Logo" width="30%"></a>
</p>


<style>
.lx-table {
  margin: 1.5em auto;
  text-align: left;
}

.lx-table caption {
  caption-side: top;
  font-size: 0.9em;
  margin-bottom: 0.6em;
  text-align: center;
}

.lx-figure {
  margin: 1.5em auto;
  text-align: center;
}

.lx-figure figcaption {
  font-size: 0.9em;
  margin-top: 0.6em;
  text-align: center;
}

table {
  margin: 0 auto 1.5em;
}

p:has(> a[id^='table-']) {
  margin: 0;
}

p:has(> a[id^='table-']) + p,
a[id^='table-'] + p {
  font-size: 0.9em;
  margin: 1.5em 0 0.6em;
  text-align: center;
}

p.lx-figure {
  margin: 1.5em auto 0;
}

p.lx-figure + p {
  font-size: 0.9em;
  margin: 0.6em 0 1.5em;
  text-align: center;
}
</style>

# IMU Rates and Configuration

Applications use IMU data on different timescales: flight control needs very frequent updates, while a display can refresh much more slowly. [Notebook 5](./5-duckiedrone-imu-architecture.ipynb) followed the measurements through the flight controller and into ROS 2, revealing several rate changes along the path. Configuring the data supplied to an application therefore requires identifying which stage controls its delivery.

This notebook investigates that choice by measuring the two IMU topics, requesting a slower MAVLink measurement stream, and comparing the resulting topic rates. The experiment connects a configuration request to its observed effect while keeping the flight controller's sensor-processing settings unchanged.

This notebook addresses the following questions:

- Which setting controls the requested rate of each IMU topic?

- How do sensor-processing rates affect the data available for transmission?

- How does a persistent PX4 parameter differ from a runtime stream-rate request?

## Configuring IMU rates and streams

[Table 1 of Notebook 5](./5-duckiedrone-imu-architecture.ipynb#table-1) lists several configurable rates; in the shipped Duckiedrone parameter file ([duckiedrone-px4-v4.params](https://docs.duckietown.com/ente/opmanual-dd24/30-duckiedrone-setup-and-config/42-flight-controller-configuration.html#load-the-preconfigured-parameter-set), at the time of writing), `IMU_GYRO_RATEMAX` is set to $800\,\mathrm{Hz}$ and `IMU_INTEG_RATE` to $250\,\mathrm{Hz}$, while PX4's onboard MAVLink profile requests $50\,\mathrm{Hz}$ for the two streams considered here.

These settings act at different stages of [the data path](./5-duckiedrone-imu-architecture.ipynb#figure-1). The first two affect processing inside PX4, whereas a MAVLink stream interval specifies how often a particular message should be sent to the Raspberry Pi. An application that needs fewer incoming messages can therefore request a longer transmission interval without changing the sensor-processing configuration.

### Setting the flight-controller parameters

The parameter `IMU_INTEG_RATE` specifies the requested rate of integrated IMU output, while `IMU_GYRO_RATEMAX` limits publication of the processed gyroscope data used by the rate controller. The IMU chip can continue sampling faster than either downstream rate.

Processing rates set how much data is available for transmission. `HIGHRES_IMU` is sent only when PX4 has a new integrated sample, so a stream requested faster than `IMU_INTEG_RATE` still delivers at most about $250\,\mathrm{Hz}$.

Both settings can be saved on the flight controller through QGroundControl. Changes to these two parameters require a reboot to take effect, so adjusting them changes the processing configuration used after startup. The experiment below leaves them unchanged.

The [PX4 parameter reference](https://docs.px4.io/main/en/advanced_config/parameter_reference.html) describes the editable parameters and their constraints; select the documentation version matching the installed firmware.

### What actually crosses the link

The two MAVLink messages used in this experiment supply different parts of the ROS 2 interface. `HIGHRES_IMU` carries processed acceleration and angular velocity, while `ATTITUDE_QUATERNION` carries the estimated orientation and body angular velocity.

MAVROS2 publishes `/mavros/imu/data_raw` when it handles `HIGHRES_IMU`. It publishes `/mavros/imu/data` when it handles `ATTITUDE_QUATERNION`, combining that message's orientation and angular velocity with the latest cached acceleration.

Consequently, an accepted request to slow a working `HIGHRES_IMU` stream should reduce the arrival rate of `/data_raw` while allowing `/data` to continue following the attitude stream. The acceleration field in successive `/data` messages may then reuse the same cached measurement.

[Table 1](#table-1) identifies the message IDs needed to request these intervals.

<a id="table-1"></a>

Table 1: MAVLink messages and their corresponding ROS 2 publication triggers in this data path.

| MAVLink message | Message ID | ROS 2 topic triggered | Relevant contents |
| --- | ---: | --- | --- |
| `HIGHRES_IMU` | 105 | `/mavros/imu/data_raw` | Processed acceleration and angular velocity |
| `ATTITUDE_QUATERNION` | 31 | `/mavros/imu/data` | Orientation and angular velocity, combined by MAVROS with cached acceleration |

[MAVLink Messaging](https://docs.px4.io/main/en/middleware/mavlink.html) explains how PX4 configures streams on its communication links.

### Asking for a different rate while everything is running

The Raspberry Pi companion computer can request a stream interval through the MAVLink command `MAV_CMD_SET_MESSAGE_INTERVAL`, whose command number is 511, while PX4 is running. MAVROS2 exposes this command through the ROS 2 service `/mavros/cmd/command`.

The following experiment uses a disarmed Duckiedrone and two terminals connected to its MAVROS2 container. It first establishes a baseline, then requests $10\,\mathrm{Hz}$ for `HIGHRES_IMU`, compares the topics, and restores the stream default.

#### Reach the ROS 2 environment

Use the environment prepared through the [LX setup instructions](../README.md#run-this-lx). From each of two base-station terminals, connect to the Duckiedrone.

For a physical Duckiedrone named `amelia` (replace `amelia` with the configured hostname), run:

```bash
ssh duckie@amelia.local
```

For a virtual Duckiedrone named `amelia` (replace `amelia` with its name), use:

```bash
dts duckiebot virtual connect amelia
```

Note that a virtual Duckiedrone must be running (`dts duckiebot virtual start amelia`) and [attached to an entity in the Duckiematrix](https://docs.duckietown.com/ente/opmanual-dd24/50-learning-experiences/supported-lxs/sensors-inertial-measurement-unit-imu.html#testing-with-the-duckiematrix) before simulated sensor data is available.

After connecting, list the running containers:

```bash
docker ps --format '{{.Names}}'
```

Identify the MAVROS2 container in the output. With the usual name `ros2-mavros`, enter it in both terminals:

```bash
docker exec -it ros2-mavros bash
```

If the listing shows a different name for that container, use the listed name. The remaining commands run inside these container shells, where the Duckiedrone's ROS 2 environment is available.

#### Establish the baseline

On a physical Duckiedrone, confirm that the flight controller is connected to the Raspberry Pi by USB; the virtual setup uses UDP instead. In terminal A, measure `/mavros/imu/data`:

```bash
ros2 topic hz /mavros/imu/data
```

Allow several reports to appear and note the approximate average rate, then stop the measurement with `Ctrl-C`. Repeat for `/data_raw`, leaving this second measurement running:

```bash
ros2 topic hz /mavros/imu/data_raw
```

With the standard onboard stream defaults, both rates should be near $50\,\mathrm{Hz}$. These are arrivals measured at this process. A configured rate is only a target: source updates, the link, and scheduling also affect what a subscriber receives, so the rate has to be measured.

#### Request a slower measurement stream

The command expresses the interval between messages in microseconds: $10\,\mathrm{Hz}$ is an interval of $0.1\,\mathrm{s}$, or $100000\,\mu\mathrm{s}$. In terminal B, request that interval for message ID 105:

```bash
ros2 service call /mavros/cmd/command mavros_msgs/srv/CommandLong \
  "{command: 511, param1: 105.0, param2: 100000.0}"
```

Here, `command` selects the interval-setting operation, `param1` selects `HIGHRES_IMU`, and `param2` supplies the interval. The remaining request fields use their default values.


```text
root@amelia:/code/src/dt-ros2-interface# ros2 service call /mavros/cmd/command mavros_msgs/srv/CommandLong "{command: 511, param1: 105.0, param2: 100000.0}"
requester: making request: mavros_msgs.srv.CommandLong_Request(broadcast=False, command=511, confirmation=0, param1=105.0, param2=100000.0, param3=0.0, param4=0.0, param5=0.0, param6=0.0, param7=0.0)

response:
mavros_msgs.srv.CommandLong_Response(success=True, result=0)
```

The service response includes `success` and `result`. A response with `success: true` and `result: 0` indicates acceptance of the command. The rate measurement in terminal A provides the separate evidence that the requested change of frequency of messages in `/mavros/imu/data_raw` took effect.

#### Compare the two topics

In terminal A, stop and restart the `/data_raw` measurement so its averaging window contains only arrivals after the change:

```bash
ros2 topic hz /mavros/imu/data_raw
```

Its measured rate should approach $10\,\mathrm{Hz}$. After several reports, stop it with `Ctrl-C` and measure the attitude topic again:

```bash
ros2 topic hz /mavros/imu/data
```

Because the request changed only `HIGHRES_IMU`, `/data` should remain near its baseline rate when the attitude stream and source remain unchanged. Its acceleration field can nevertheless update less frequently because MAVROS receives new acceleration through the slower measurement stream.

#### Restore the stream

After stopping the measurement with `Ctrl-C`, return `HIGHRES_IMU` to its configured default by sending an interval of zero in terminal B:

```bash
ros2 service call /mavros/cmd/command mavros_msgs/srv/CommandLong \
  "{command: 511, param1: 105.0, param2: 0.0}"
```

Check the response, then measure `/data_raw` again in terminal A to confirm that its rate returns near the original baseline:

```bash
ros2 topic hz /mavros/imu/data_raw
```

Stop the measurement with `Ctrl-C` after the comparison.

These interval requests affect the running MAVLink connection and do not save a persistent PX4 parameter. After a flight-controller reboot, startup configuration and any subsequent requests establish the stream rates again.

The [MAVLink common message set](https://mavlink.io/en/messages/common.html) defines the message IDs and command parameters. For this command, a positive interval requests a period in microseconds, zero requests the default, and `-1` disables the selected stream.

### When the rate does not change

- If `ros2 topic hz` prints no reports, check the connection to the Duckiedrone and that the MAVROS2 container is running before changing a stream. Without a working baseline, a later measurement cannot show the effect of the request.

- If the service call keeps waiting, stop it with `Ctrl-C` and check that `/mavros/cmd/command` appears in `ros2 service list`.

- If the response reports `success=False`, the requested interval is not active.

- If the request is accepted but the measured rate differs from the requested one, check whether PX4 produces new samples that often, whether the link can carry them, and whether another process has requested a different interval.

### One switch that looks like a rate and is not

MAVROS2 also has a plugin configuration in `px4_config.yaml`. Its `imu` plugin handles the incoming IMU-related messages and publishes the ROS 2 topics used in this experiment.

Excluding that plugin prevents it from creating those topics, while PX4 can continue transmitting the corresponding MAVLink messages. Plugin selection therefore determines which ROS interfaces MAVROS provides; reducing traffic from the flight controller requires changing the relevant stream configuration.

## Summary and next steps

The experiment performed in this notebook shows the separation between a requested transmission rate and the measured delivery of IMU messages. Changing the `HIGHRES_IMU` interval affects `/mavros/imu/data_raw`, while `/mavros/imu/data` follows the attitude stream and can reuse acceleration received earlier.

PX4's persistent sensor-processing parameters, runtime MAVLink interval requests, and MAVROS plugin selection therefore control different parts of the data path. Identifying the relevant stage makes it possible to change message delivery without unnecessarily changing the processing used by the flight controller.

[Notebook 7](./7-interfacing-with-imu.ipynb) uses this ROS 2 interface to inspect individual messages and implement a subscriber that reports their contents.

## Further reading

[Finding/Updating Parameters](https://docs.px4.io/main/en/advanced_config/parameters.html) shows how to read and change PX4 parameters in QGroundControl.

## Checkpoint

Run the self-check in the next cell. Write or select a response before revealing the answer.


In [ ]:
import sys
from pathlib import Path

working_directory = Path.cwd()
parent_directory = working_directory.parent
if (parent_directory / "packages").is_dir():
    parent_directory_path = str(parent_directory)
    sys.path.insert(0, parent_directory_path)

from packages.checkpoint_self_check import display_checkpoint_self_checks

display_checkpoint_self_checks()